## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-09-13-49-00 +0000,wsj,Crédit Agricole Proposes Merger of Italian Bus...,https://www.wsj.com/business/deals/credit-agri...
1,2026-11-09-13-48-03 +0000,nyt,Bipartisan Support for Israel Erodes as Both S...,https://www.nytimes.com/2026/11/09/us/politics...
2,2026-11-09-13-47-52 +0000,cbc,Nvidia's shares slide as chip giant probes CEO...,https://www.cbc.ca/news/business/nvidia-shares...
3,2026-11-09-13-44-00 +0000,wsj,Children’s Place Files for Bankruptcy With Pla...,https://www.wsj.com/business/retail/childrens-...
4,2026-11-09-13-42-03 +0000,bbc,Rising cost of living tests UK's net zero ambi...,https://www.bbc.co.uk/news/videos/cm1dwd4e4ywv...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2705/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
213,trump,38
18,nvidia,23
409,white,16
252,was,14
24,ceo,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
178,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...,159
177,2026-11-09-00-31-25 +0000,wapo,Trump says he was ‘only kidding’ about offerin...,https://www.washingtonpost.com/politics/2026/1...,123
187,2026-11-08-23-56-01 +0000,cbc,Trump insists he was 'only kidding' about want...,https://www.cbc.ca/news/world/trump-dana-white...,120
61,2026-11-09-11-26-13 +0000,nypost,Trump says ‘I was only kidding’ with press sec...,https://nypost.com/2026/11/09/us-news/trump-sa...,115
109,2026-11-09-08-49-17 +0000,bbc,Trump says he was 'kidding' about Dana White p...,https://www.bbc.co.uk/news/articles/ck0j0j8795...,111


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
178,159,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...
224,90,2026-11-08-20-52-22 +0000,nypost,Nvidia CEO faces probe over affair with allege...,https://nypost.com/2026/11/08/tech/jensen-huan...
160,65,2026-11-09-02-00-00 +0000,wsj,Voters fed up with high prices gave Democrats ...,https://www.wsj.com/politics/elections/democra...
291,49,2026-11-08-14-07-36 +0000,nypost,Netanyahu’s rival will have first shot at beco...,https://nypost.com/2026/11/08/us-news/israels-...
182,42,2026-11-09-00-23-00 +0000,wsj,Two former senior Trump advisers have been sen...,https://www.wsj.com/politics/policy/former-tru...
28,40,2026-11-09-12-38-00 +0000,wsj,Jensen Huang Investigation Rocks Nvidia as Boa...,https://www.wsj.com/cfo-journal/jensen-huang-i...
107,37,2026-11-09-09-24-53 +0000,nyt,"Suspect in Mass Shooting in Krakow, Poland, Di...",https://www.nytimes.com/2026/11/08/world/europ...
205,33,2026-11-08-22-05-54 +0000,nypost,Saudi Aramco tower damaged in Houthi missiles ...,https://nypost.com/2026/11/08/world-news/saudi...
168,32,2026-11-09-01-30-37 +0000,nypost,California dad stabs wife to death in front of...,https://nypost.com/2026/11/08/us-news/californ...
228,31,2026-11-08-20-04-00 +0000,wsj,What’s on tap as Congress returns to Washingto...,https://www.wsj.com/politics/policy/defeated-g...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
